# QTAP: Quantum Torsion Angle Predictor
### Variational Quantum Born Machines for Ramachandran phi/psi Distribution Prediction
**Author:** Tommaso R. Marena, Catholic University of America, 2026

**v3 — publication-ready:**
- Real phi/psi data from RCSB PDB REST API (X-ray, <=2.0 A)
- 6 qubits -> 64 bins (8x8 grid, 45 deg resolution)
- 6 physicochemical features including helix propensity (matches NQ)
- Fair 80/20 structure-level train/test split
- 4 classical baselines: KDE, von Mises mixture, RBM, MLP (LOO)
- Bootstrap 95% CIs on KL and JS
- Circuit depth ablation
- All 20 amino acids

In [ ]:
import subprocess, sys
pkgs = ['qiskit>=1.0.0','qiskit-aer>=0.14.0','scipy>=1.11',
        'matplotlib','pandas','torch','tqdm','seaborn',
        'biopython>=1.81','requests','scikit-learn']
for p in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('All dependencies installed.')

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
import requests, json, os, time, warnings
from io import StringIO
from collections import defaultdict
from scipy.optimize import minimize
from scipy.special import rel_entr
from scipy.spatial.distance import jensenshannon
from sklearn.neighbors import KernelDensity
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from tqdm.notebook import tqdm
from Bio.PDB import PDBParser, PPBuilder
warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42)

NQ        = 6        # qubits; also == number of AA features
N_FEAT    = NQ       # must stay equal
DEPTH     = 3
SHOTS     = 2048
STEPS     = 400
N_RESTART = 3
N_BOOT    = 500
N_PDB     = 120
MIN_ANG   = 30
NB        = 2**NQ    # 64
GRID      = 2**(NQ//2)  # 8
AA        = list('ACDEFGHIKLMNPQRSTVWY')
sim       = AerSimulator()
os.makedirs('qtap_outputs', exist_ok=True)
print(f'NQ={NQ}, NB={NB}, GRID={GRID}x{GRID}')

## Cell 3 — Fetch Real phi/psi Data from RCSB PDB

In [ ]:
CACHE = 'qtap_outputs/pdb_torsions.json'

def query_pdb_ids(n=100):
    url = 'https://search.rcsb.org/rcsbsearch/v2/query'
    q = {'query':{'type':'group','logical_operator':'and','nodes':[
            {'type':'terminal','service':'text','parameters':{
                'attribute':'exptl.method','operator':'exact_match','value':'X-RAY DIFFRACTION'}},
            {'type':'terminal','service':'text','parameters':{
                'attribute':'rcsb_entry_info.resolution_combined','operator':'less_or_equal','value':2.0}},
            {'type':'terminal','service':'text','parameters':{
                'attribute':'entity_poly.rcsb_entity_polymer_type','operator':'exact_match','value':'Protein'}}
        ]},
        'return_type':'entry',
        'request_options':{'paginate':{'start':0,'rows':n},
                           'sort':[{'sort_by':'score','direction':'desc'}]}}
    r = requests.post(url, json=q, timeout=30); r.raise_for_status()
    return [x['identifier'] for x in r.json().get('result_set',[])]

def fetch_torsions(pdb_id):
    r = requests.get(f'https://files.rcsb.org/download/{pdb_id}.pdb', timeout=20)
    if r.status_code != 200: return {}
    struct = PDBParser(QUIET=True).get_structure(pdb_id, StringIO(r.text))
    out = defaultdict(list)
    for model in struct:
        for chain in model:
            for pp in PPBuilder().build_peptides(chain):
                for res, (phi, psi) in zip(str(pp.get_sequence()), pp.get_phi_psi_list()):
                    if phi is not None and psi is not None:
                        out[res].append((np.degrees(phi), np.degrees(psi)))
    return dict(out)

if os.path.exists(CACHE):
    with open(CACHE) as f: raw = json.load(f)
    print(f'Loaded cache: {CACHE}')
else:
    ids = query_pdb_ids(N_PDB)
    print(f'Fetching {len(ids)} structures...')
    raw = defaultdict(list)
    for pid in tqdm(ids, desc='PDB fetch'):
        for aa, angs in fetch_torsions(pid).items(): raw[aa].extend(angs)
        time.sleep(0.05)
    raw = dict(raw)
    with open(CACHE,'w') as f: json.dump(raw, f)
    print(f'Cached -> {CACHE}')

for aa in AA: print(f'  {aa}: {len(raw.get(aa,[]))} angles')

In [ ]:
EDGES = np.linspace(-180, 180, GRID + 1)

def build_hist(angles):
    if len(angles) == 0: return None
    arr = np.array(angles)
    H, _, _ = np.histogram2d(arr[:,0], arr[:,1], bins=[EDGES, EDGES])
    H = H.T
    H += 1e-3
    return (H / H.sum()).ravel()

refs, valid_aa = {}, []
for aa in AA:
    angles = raw.get(aa, [])
    if len(angles) >= MIN_ANG:
        refs[aa] = build_hist(angles)
        valid_aa.append(aa)

print(f'AAs with >={MIN_ANG} angles: {valid_aa}')
for aa in valid_aa:
    assert (refs[aa] > 0).all() and abs(refs[aa].sum()-1.0) < 1e-4
print('All histograms valid.')

In [ ]:
rng = np.random.RandomState(42)
refs_train, refs_test = {}, {}
for aa in valid_aa:
    angles = np.array(raw[aa])
    idx = rng.permutation(len(angles))
    split = int(0.8 * len(angles))
    tr, te = angles[idx[:split]], angles[idx[split:]]
    if len(tr) >= MIN_ANG and len(te) >= MIN_ANG // 4:
        refs_train[aa] = build_hist(tr.tolist())
        refs_test[aa]  = build_hist(te.tolist())

split_aa   = list(refs_train.keys())
refs_eval  = refs_test
print(f'Split AAs: {split_aa} ({len(split_aa)} total)')

In [ ]:
# 6 features: mol_weight, hydrophobicity, pKa, charge, aromaticity, helix_propensity
# Helix propensity from Pace & Scholtz (1998) Biophys J 75:422
AA_PROPS = {
    'A':(89.09,  1.8, 6.00,  0.0, 0.0,  1.00),
    'C':(121.16, 2.5, 5.07,  0.0, 0.0,  0.68),
    'D':(133.10,-3.5, 2.77, -1.0, 0.0,  0.40),
    'E':(147.13,-3.5, 3.22, -1.0, 0.0,  0.59),
    'F':(165.19, 2.8, 5.48,  0.0, 1.0,  0.54),
    'G':( 75.03,-0.4, 5.97,  0.0, 0.0,  0.00),
    'H':(155.16,-3.2, 7.59,  0.1, 1.0,  0.41),
    'I':(131.17, 4.5, 6.02,  0.0, 0.0,  0.41),
    'K':(146.19,-3.9,10.53,  1.0, 0.0,  0.26),
    'L':(131.17, 3.8, 5.98,  0.0, 0.0,  0.79),
    'M':(149.21, 1.9, 5.74,  0.0, 0.0,  0.73),
    'N':(132.12,-3.5, 5.41,  0.0, 0.0,  0.21),
    'P':(115.13,-1.6, 6.30,  0.0, 0.0, -0.99),
    'Q':(146.15,-3.5, 5.65,  0.0, 0.0,  0.39),
    'R':(174.20,-4.5,10.76,  1.0, 0.0,  0.21),
    'S':(105.09,-0.8, 5.68,  0.0, 0.0,  0.11),
    'T':(119.12,-0.7, 5.60,  0.0, 0.0,  0.09),
    'V':(117.15, 4.2, 5.96,  0.0, 0.0,  0.14),
    'W':(204.23,-0.9, 5.89,  0.0, 1.0,  0.49),
    'Y':(181.19,-1.3, 5.66,  0.0, 1.0,  0.30),
}
RANGES = [
    ( 75.03, 204.23),  # mol_weight
    ( -4.5,    4.5 ),  # hydrophobicity
    (  2.77,  10.76),  # pKa
    ( -1.0,    1.0 ),  # charge
    (  0.0,    1.0 ),  # aromaticity
    ( -0.99,   1.00),  # helix_propensity
]
assert len(RANGES) == NQ, f'len(RANGES)={len(RANGES)} must equal NQ={NQ}'
assert all(len(v)==NQ for v in AA_PROPS.values()), 'Each AA_PROPS entry must have NQ=6 values'

def encode(a):
    lo, hi = zip(*RANGES)
    return np.array([(AA_PROPS[a][i]-lo[i])/(hi[i]-lo[i]+1e-9)*2*np.pi for i in range(NQ)])

enc = {a: encode(a) for a in AA}
for a in AA: assert enc[a].shape == (NQ,)
print(f'Encodings ready. Shape ({NQ},) per AA.')
for a in ['G','A','P','W']: print(f'  {a}: {np.round(enc[a],3)}')

In [ ]:
def build_circuit(nq=NQ, depth=DEPTH):
    e = ParameterVector('enc', nq)
    v = ParameterVector('var', 2*nq*depth)
    qc = QuantumCircuit(nq)
    for i in range(nq): qc.h(i)
    for i in range(nq): qc.ry(e[i], i)
    k = 0
    for _ in range(depth):
        for i in range(nq): qc.ry(v[k], i); k += 1
        for i in range(nq): qc.rz(v[k], i); k += 1
        for i in range(nq-1): qc.cx(i, i+1)
        qc.cx(nq-1, 0)
    qc.measure_all()
    return qc, e, v

qc, epar, vpar = build_circuit()
N_VAR = len(vpar)
print(f'Circuit: {NQ} qubits, depth {DEPTH}, {N_VAR} variational params')
print(qc.draw(output='text', fold=100))

In [ ]:
def born(x, theta, shots=SHOTS, nq=NQ, qc_=None, epar_=None, vpar_=None):
    qc_ = qc_ or qc; epar_ = epar_ or epar; vpar_ = vpar_ or vpar
    bind = {p: float(x[i]) for i,p in enumerate(epar_)}
    bind.update({p: float(theta[i]) for i,p in enumerate(vpar_)})
    counts = sim.run(
        transpile(qc_.assign_parameters(bind), sim, optimization_level=1),
        shots=shots).result().get_counts()
    p = np.zeros(2**nq)
    for b, c in counts.items(): p[int(b,2)] += c
    p = p/p.sum() + 1e-9
    return p/p.sum()

def KL(ref, pred): return float(np.sum(rel_entr(ref, pred)))
def JS(ref, pred): return float(jensenshannon(ref, pred)**2)

th_test = np.random.uniform(0, 2*np.pi, N_VAR)
p_test  = born(enc['A'], th_test)
print(f'Forward pass OK: {NB} bins, sum={p_test.sum():.4f}')

In [ ]:
def train_one(a, ref, steps=STEPS, seed=0):
    rng2 = np.random.RandomState(seed)
    th0  = rng2.uniform(0, 2*np.pi, N_VAR)
    hist = []
    def obj(th):
        loss = KL(ref, born(enc[a], th)); hist.append(loss); return loss
    minimize(obj, th0, method='COBYLA',
             options={'maxiter':steps,'rhobeg':0.5,'catol':0})
    best_i = int(np.argmin(hist))
    return hist[best_i], th0, hist   # note: COBYLA doesn't return best x directly

def train_one_v2(a, ref, steps=STEPS, seed=0):
    rng2 = np.random.RandomState(seed)
    th0  = rng2.uniform(0, 2*np.pi, N_VAR)
    hist, best_th, best_kl = [], th0.copy(), np.inf
    def obj(th):
        nonlocal best_th, best_kl
        loss = KL(ref, born(enc[a], th))
        hist.append(loss)
        if loss < best_kl: best_kl = loss; best_th = th.copy()
        return loss
    minimize(obj, th0, method='COBYLA',
             options={'maxiter':steps,'rhobeg':0.5,'catol':0})
    return best_kl, best_th, hist

def train(a, ref, steps=STEPS, n_restart=N_RESTART):
    best_kl, best_theta, best_hist = np.inf, None, []
    for k in range(n_restart):
        kl, theta, hist = train_one_v2(a, ref, steps, seed=k*7+13)
        if kl < best_kl:
            best_kl, best_theta, best_hist = kl, theta, hist
    return {'kl': best_kl, 'theta': best_theta, 'hist': best_hist}

targets = split_aa
res = {}
for a in tqdm(targets, desc='Training QTAP'):
    res[a] = train(a, refs_train[a])
print('QTAP training done.')
for a in targets:
    ev_kl = KL(refs_eval[a], born(enc[a], res[a]['theta']))
    print(f'  {a}: train_KL={res[a]["kl"]:.4f}  eval_KL={ev_kl:.4f}')

## Classical Baselines
All trained on train-split angles, evaluated on test-split histograms.
1. KDE — Gaussian KDE with CV bandwidth
2. von Mises mixture — 3-component EM
3. RBM — classical Born machine analogue
4. MLP — leave-one-out generalisation test

In [ ]:
BIN_CENTRES = [(p,q) for q in (EDGES[:-1]+EDGES[1:])/2 for p in (EDGES[:-1]+EDGES[1:])/2]
BC = np.array(BIN_CENTRES)

kde_preds = {}
for a in tqdm(targets, desc='KDE'):
    ang = np.array(raw[a])
    idx = np.random.RandomState(42).permutation(len(ang))
    tr  = ang[idx[:int(0.8*len(ang))]]
    best_bw, best_score = 15.0, -np.inf
    for bw in np.logspace(0.5, 1.5, 8):
        sc = []
        for _ in range(3):
            ii  = np.random.choice(len(tr), int(0.8*len(tr)), replace=False)
            vii = np.setdiff1d(np.arange(len(tr)), ii)
            if len(vii)==0: continue
            sc.append(KernelDensity(bandwidth=bw,kernel='gaussian').fit(tr[ii]).score(tr[vii]))
        if sc and np.mean(sc) > best_score: best_score=np.mean(sc); best_bw=bw
    kde = KernelDensity(bandwidth=best_bw,kernel='gaussian').fit(tr)
    p   = np.exp(kde.score_samples(BC)); p += 1e-9; p /= p.sum()
    kde_preds[a] = p
print('KDE done.')

In [ ]:
def vonmises_mix_pred(angles_rad, n_comp=3, n_iter=80):
    N = len(angles_rad)
    rng2 = np.random.RandomState(0)
    mu    = angles_rad[rng2.choice(N, n_comp, replace=False)]
    kappa = np.ones(n_comp)*2.0
    pi    = np.ones(n_comp)/n_comp
    for _ in range(n_iter):
        log_r = np.zeros((N, n_comp))
        for k in range(n_comp):
            log_r[:,k] = np.log(pi[k]+1e-9) + kappa[k]*np.sum(np.cos(angles_rad-mu[k]),axis=1)
        log_r -= log_r.max(1,keepdims=True)
        r = np.exp(log_r); r /= r.sum(1,keepdims=True)
        Nk = r.sum(0)+1e-9; pi = Nk/N
        for k in range(n_comp):
            S = (r[:,k:k+1]*np.sin(angles_rad)).sum(0)
            C = (r[:,k:k+1]*np.cos(angles_rad)).sum(0)
            mu[k] = np.arctan2(S,C)
            R = np.sqrt(S**2+C**2)/Nk[k]
            kappa[k] = np.clip(R*(2-R**2)/(1-R**2+1e-9), 0.1, 20.0)
    bc_rad = np.radians(BC)
    lp = np.zeros(NB)
    for k in range(n_comp):
        lp += pi[k]*np.exp(kappa[k]*np.sum(np.cos(bc_rad-mu[k]),axis=1))
    lp += 1e-9; return lp/lp.sum()

vm_preds = {}
for a in tqdm(targets, desc='vonMises mix'):
    ang = np.array(raw[a])
    idx = np.random.RandomState(42).permutation(len(ang))
    tr  = np.radians(ang[idx[:int(0.8*len(ang))]])
    vm_preds[a] = vonmises_mix_pred(tr)
print('von Mises mixture done.')

In [ ]:
class RBM(nn.Module):
    def __init__(self, n_vis=NB, n_hid=32):
        super().__init__()
        self.W  = nn.Parameter(torch.randn(n_hid,n_vis)*0.01)
        self.bv = nn.Parameter(torch.zeros(n_vis))
        self.bh = nn.Parameter(torch.zeros(n_hid))
    def free_energy(self,v):
        return -(v*self.bv).sum(-1) - torch.log(1+torch.exp(v@self.W.t()+self.bh)).sum(-1)
    def probs(self):
        fe = self.free_energy(torch.eye(NB))
        return torch.softmax(-fe,dim=0).detach().numpy()

rbm_preds = {}
for a in tqdm(targets, desc='RBM'):
    rbm = RBM(); opt = torch.optim.Adam(rbm.parameters(), lr=3e-3)
    ref_t = torch.tensor(refs_train[a], dtype=torch.float32)
    for ep in range(2000):
        opt.zero_grad()
        p = torch.tensor(rbm.probs(), dtype=torch.float32)
        torch.sum(ref_t*torch.log(ref_t/(p+1e-9))).backward(); opt.step()
    rbm_preds[a] = rbm.probs()
print('RBM done.')

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_FEAT,128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128,128),    nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, NB),    nn.Softmax(dim=-1))
    def forward(self,x): return self.net(x)

def feat(a):
    lo,hi = zip(*RANGES)
    return [(AA_PROPS[a][i]-lo[i])/(hi[i]-lo[i]+1e-9) for i in range(N_FEAT)]

mlp_preds = {}
for test_a in tqdm(targets, desc='MLP (LOO)'):
    train_aas = [a for a in split_aa if a != test_a]
    X_tr = torch.tensor([feat(a) for a in train_aas], dtype=torch.float32)
    Y_tr = torch.tensor(np.array([refs_train[a] for a in train_aas]), dtype=torch.float32)
    X_te = torch.tensor([feat(test_a)], dtype=torch.float32)
    mlp = MLP(); opt = torch.optim.Adam(mlp.parameters(), lr=5e-4, weight_decay=1e-4)
    lossfn = nn.KLDivLoss(reduction='batchmean')
    for ep in range(3000):
        mlp.train(); opt.zero_grad()
        loss = lossfn(torch.log(mlp(X_tr)+1e-9), Y_tr)
        loss.backward(); opt.step()
    mlp.eval()
    with torch.no_grad(): mlp_preds[test_a] = mlp(X_te).numpy()[0]
print('MLP LOO done.')

In [ ]:
def bootstrap_ci(pred, angles_test, n_boot=N_BOOT, alpha=0.05):
    kl_s, js_s = [], []
    n = len(angles_test)
    rng3 = np.random.RandomState(0)
    for _ in range(n_boot):
        idx = rng3.choice(n, n, replace=True)
        rb  = build_hist(angles_test[idx].tolist())
        kl_s.append(KL(rb, pred)); js_s.append(JS(rb, pred))
    lo, hi = alpha/2, 1-alpha/2
    return {'kl_mean':np.mean(kl_s),'kl_lo':np.quantile(kl_s,lo),'kl_hi':np.quantile(kl_s,hi),
            'js_mean':np.mean(js_s),'js_lo':np.quantile(js_s,lo),'js_hi':np.quantile(js_s,hi)}

methods = {
    'QTAP':     lambda a: born(enc[a], res[a]['theta'], shots=SHOTS*2),
    'KDE':      lambda a: kde_preds[a],
    'vonMises': lambda a: vm_preds[a],
    'RBM':      lambda a: rbm_preds[a],
    'MLP':      lambda a: mlp_preds[a],
}
rows = []
for a in tqdm(targets, desc='Bootstrap CIs'):
    ang_te = np.array(raw[a])
    idx = np.random.RandomState(42).permutation(len(ang_te))
    ang_te = ang_te[idx[int(0.8*len(ang_te)):]]
    for mn, mf in methods.items():
        ci = bootstrap_ci(mf(a), ang_te)
        rows.append([a, mn,
                     round(ci['kl_mean'],4),round(ci['kl_lo'],4),round(ci['kl_hi'],4),
                     round(ci['js_mean'],4),round(ci['js_lo'],4),round(ci['js_hi'],4)])
df = pd.DataFrame(rows, columns=['Residue','Method',
    'KL_mean','KL_lo','KL_hi','JS_mean','JS_lo','JS_hi'])
print(df.to_string(index=False))
df.to_csv('qtap_outputs/results_bootstrap.csv', index=False)
df

In [ ]:
ablation_aa = targets[:4]
DEPTHS      = [1, 2, 3, 4]
ablation_rows = []
for d in tqdm(DEPTHS, desc='Depth ablation'):
    qc_d, ep_d, vp_d = build_circuit(nq=NQ, depth=d)
    for a in ablation_aa:
        best_kl = np.inf
        for k in range(2):
            th0 = np.random.RandomState(k*3+7).uniform(0,2*np.pi,len(vp_d))
            bkl, _, _ = train_one_v2.__wrapped__(a, refs_train[a], 200, k*3+7) if hasattr(train_one_v2,'__wrapped__') else train_one_v2(a, refs_train[a], 200, k*3+7)
            if bkl < best_kl: best_kl = bkl
        ablation_rows.append([d, len(vp_d), a, round(best_kl,4)])
df_abl = pd.DataFrame(ablation_rows, columns=['Depth','N_params','Residue','Train_KL'])
print(df_abl.to_string(index=False))
df_abl.to_csv('qtap_outputs/ablation_depth.csv', index=False)
df_abl

In [ ]:
# --- Convergence curves ---
n_plot = min(len(targets), 6)
fig, axs = plt.subplots(1, n_plot, figsize=(3.5*n_plot, 3.5))
if n_plot == 1: axs = [axs]
for ax, a in zip(axs, targets[:n_plot]):
    ax.plot(res[a]['hist'], lw=1.2, color='steelblue')
    ax.axhline(res[a]['kl'], ls='--', color='crimson', lw=1, label=f"best={res[a]['kl']:.3f}")
    ax.set_title(a); ax.set_xlabel('COBYLA call'); ax.set_ylabel('KL'); ax.legend(fontsize=8)
plt.suptitle('QTAP Training Convergence', y=1.02)
plt.tight_layout()
plt.savefig('qtap_outputs/convergence.png', dpi=150, bbox_inches='tight'); plt.show()

# --- Summary bar chart ---
method_order = ['QTAP','KDE','vonMises','RBM','MLP']
pivot = df.groupby('Method')['JS_mean'].mean().reindex(method_order)
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(method_order, pivot.values,
             color=['#1a6faf','#e07b39','#3aaa35','#8e44ad','#c0392b'],
             edgecolor='k', linewidth=0.6)
for bar,v in zip(bars,pivot.values): ax.text(bar.get_x()+bar.get_width()/2, v+0.001, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Mean JS Divergence'); ax.set_title('QTAP vs Classical Baselines')
plt.tight_layout(); plt.savefig('qtap_outputs/summary_bar.png', dpi=150); plt.show()

# --- Ramachandran heatmaps ---
labels = [f'{int(e)}' for e in (EDGES[:-1]+EDGES[1:])/2]
for a in targets:
    grids  = [refs_eval[a], born(enc[a],res[a]['theta'],shots=SHOTS*4),
              kde_preds[a], vm_preds[a], rbm_preds[a], mlp_preds[a]]
    grids  = [g.reshape(GRID,GRID) for g in grids]
    titles = ['Reference (PDB)','QTAP','KDE','vonMises','RBM','MLP']
    fig, axs2 = plt.subplots(1,6,figsize=(22,3.5))
    fig.suptitle(f'{a}  |  QTAP KL={KL(refs_eval[a],born(enc[a],res[a]["theta"])):.3f}', fontsize=11)
    for ax,g,t in zip(axs2,grids,titles):
        im = ax.imshow(g,origin='lower',cmap='hot_r',vmin=0,vmax=g.max())
        ax.set_title(t,fontsize=9); step=max(1,GRID//4)
        ax.set_xticks(range(0,GRID,step)); ax.set_xticklabels(labels[::step],fontsize=7)
        ax.set_yticks(range(0,GRID,step)); ax.set_yticklabels(labels[::step],fontsize=7)
        ax.set_xlabel('phi',fontsize=8); ax.set_ylabel('psi',fontsize=8)
        plt.colorbar(im,ax=ax,fraction=0.046)
    plt.tight_layout(); plt.savefig(f'qtap_outputs/rama_{a}.png',dpi=150,bbox_inches='tight'); plt.show()
print('All figures saved.')

In [ ]:
latex = df.pivot_table(index='Residue',columns='Method',values='JS_mean')[method_order]
with open('qtap_outputs/table_js.tex','w') as f: f.write(latex.to_latex(float_format='%.4f',bold_rows=True))
print('Saved: results_bootstrap.csv, ablation_depth.csv, table_js.tex')
print()
print('=== FINAL RESULTS (JS divergence) ===')
print(latex.round(4).to_string())